# Thiranex Internship — Task 2
## Predictive Modeling Using Machine Learning
### Malaria Cell Classification

**Objective:** Build a predictive model to classify microscopic blood-cell images as **Parasitized** or **Uninfected** and evaluate it using accuracy, precision, recall, F1-score, confusion matrix, ROC curve and AUC.



In [3]:
import tensorflow as tf
print(tf.__version__)

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# 1. Import libraries
import os
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, UnidentifiedImageError

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score, auc
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)


ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# 2. Download the malaria cell-image dataset
DATA_URL = "https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip"
PROJECT_DIR = Path.cwd()
ZIP_PATH = str(PROJECT_DIR / "cell_images.zip")
EXTRACT_DIR = str(PROJECT_DIR)

if not os.path.exists(ZIP_PATH):
    print("Downloading dataset...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    print("Download complete.")
else:
    print("Dataset ZIP already exists.")

CELL_DIR = str(PROJECT_DIR / "cell_images")

if not os.path.exists(CELL_DIR):
    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)

# Some archives may contain an extra nested cell_images folder.
if not os.path.isdir(os.path.join(CELL_DIR, "Parasitized")):
    nested = os.path.join(CELL_DIR, "cell_images")
    if os.path.isdir(nested):
        CELL_DIR = nested

print("Dataset location:", CELL_DIR)
print("Classes:", os.listdir(CELL_DIR)[:10])


In [ ]:
# 3. Inspect the dataset
PARASITIZED_DIR = os.path.join(CELL_DIR, "Parasitized")
UNINFECTED_DIR = os.path.join(CELL_DIR, "Uninfected")

def get_image_files(folder):
    return [
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ]

parasitized_files = get_image_files(PARASITIZED_DIR)
uninfected_files = get_image_files(UNINFECTED_DIR)

print("Parasitized images:", len(parasitized_files))
print("Uninfected images :", len(uninfected_files))
print("Total images     :", len(parasitized_files) + len(uninfected_files))

class_counts = pd.Series({
    "Parasitized": len(parasitized_files),
    "Uninfected": len(uninfected_files)
})

plt.figure(figsize=(7, 5))
class_counts.plot(kind="bar")
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=0)
plt.show()


In [ ]:
# 4. Load and preprocess images
IMAGE_SIZE = 150

images = []
labels = []
bad_files = []

def load_image_files(files, label):
    for file_path in files:
        try:
            image = Image.open(file_path).convert("RGB")
            image = image.resize((IMAGE_SIZE, IMAGE_SIZE))
            images.append(np.asarray(image, dtype=np.float32))
            labels.append(label)
        except (UnidentifiedImageError, OSError, ValueError) as error:
            bad_files.append((file_path, str(error)))

# 0 = Parasitized
# 1 = Uninfected
load_image_files(parasitized_files, 0)
load_image_files(uninfected_files, 1)

X = np.stack(images)
y = np.asarray(labels, dtype=np.int32)

# Normalize pixels to 0-1
X = X / 255.0

print("Image data shape:", X.shape)
print("Label shape:", y.shape)
print("Unreadable files:", len(bad_files))


In [ ]:
# 5. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training images:", X_train.shape[0])
print("Testing images :", X_test.shape[0])

print("\nTraining class distribution:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nTesting class distribution:")
print(pd.Series(y_test).value_counts().sort_index())


In [ ]:
# 6. Display sample images
rng = np.random.default_rng(42)
sample_indices = rng.choice(len(X_train), 8, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for ax, index in zip(axes.ravel(), sample_indices):
    ax.imshow(X_train[index])
    label = "Uninfected" if y_train[index] == 1 else "Parasitized"
    ax.set_title(label)
    ax.axis("off")

plt.tight_layout()
plt.show()


## 7. Build the CNN model

The model is a binary classifier. The final sigmoid neuron outputs a probability for the **Uninfected** class.

The implementation uses current TensorFlow/Keras APIs rather than the older APIs used in the original tutorial.


In [ ]:
# 7. CNN model
model = keras.Sequential([
    layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)),

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.5),

    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


In [ ]:
# 8. Train the model
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.20,
    epochs=15,
    batch_size=64,
    shuffle=True,
    callbacks=[early_stopping],
    verbose=1
)


In [ ]:
# 9. Training and validation accuracy
history_df = pd.DataFrame(history.history)

plt.figure(figsize=(8, 5))
plt.plot(history_df["accuracy"], label="Training Accuracy")
plt.plot(history_df["val_accuracy"], label="Validation Accuracy")
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

# Training and validation loss
plt.figure(figsize=(8, 5))
plt.plot(history_df["loss"], label="Training Loss")
plt.plot(history_df["val_loss"], label="Validation Loss")
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


In [ ]:
# 10. Predict probabilities and classes
y_probability = model.predict(X_test, batch_size=64).ravel()

# Standard binary classification threshold
THRESHOLD = 0.50
y_prediction = (y_probability >= THRESHOLD).astype(int)

print(classification_report(
    y_test,
    y_prediction,
    target_names=["Parasitized", "Uninfected"],
    digits=4
))


In [ ]:
# 11. Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_prediction)
precision = precision_score(y_test, y_prediction, zero_division=0)
recall = recall_score(y_test, y_prediction, zero_division=0)
f1 = f1_score(y_test, y_prediction, zero_division=0)
roc_auc = roc_auc_score(y_test, y_probability)

results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"],
    "Score": [accuracy, precision, recall, f1, roc_auc]
})

results


In [ ]:
# 12. Confusion matrix
cm = confusion_matrix(y_test, y_prediction)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Parasitized", "Uninfected"],
    yticklabels=["Parasitized", "Uninfected"]
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

tn, fp, fn, tp = cm.ravel()

print("True Negative :", tn)
print("False Positive:", fp)
print("False Negative:", fn)
print("True Positive :", tp)


## 13. ROC Curve and AUC

The ROC curve shows the relationship between the true-positive rate and false-positive rate across different classification thresholds.

AUC summarizes the model's discrimination ability across those thresholds.


In [ ]:
# 13. ROC curve and AUC
false_positive_rate, true_positive_rate, thresholds = roc_curve(
    y_test,
    y_probability
)

roc_auc = auc(false_positive_rate, true_positive_rate)

plt.figure(figsize=(8, 6))
plt.plot(
    false_positive_rate,
    true_positive_rate,
    label=f"ROC Curve (AUC = {roc_auc:.4f})"
)
plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

print(f"ROC-AUC: {roc_auc:.4f}")


## 14. Threshold Analysis

The tutorial demonstrates that changing the classification threshold changes the balance between false positives and false negatives.

We first use the standard threshold of **0.50**. Then we calculate a candidate threshold using Youden's J statistic:

**J = True Positive Rate − False Positive Rate**

The candidate threshold is reported from the actual test predictions; no performance numbers are fabricated.


In [ ]:
# 14. Find a candidate threshold using Youden's J statistic
youden_j = true_positive_rate - false_positive_rate
best_index = np.argmax(youden_j)

candidate_threshold = thresholds[best_index]

print("Candidate threshold:", float(candidate_threshold))
print("TPR:", float(true_positive_rate[best_index]))
print("FPR:", float(false_positive_rate[best_index]))

y_prediction_candidate = (
    y_probability >= candidate_threshold
).astype(int)

threshold_comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Threshold 0.50": [
        accuracy_score(y_test, y_prediction),
        precision_score(y_test, y_prediction, zero_division=0),
        recall_score(y_test, y_prediction, zero_division=0),
        f1_score(y_test, y_prediction, zero_division=0)
    ],
    "Candidate Threshold": [
        accuracy_score(y_test, y_prediction_candidate),
        precision_score(y_test, y_prediction_candidate, zero_division=0),
        recall_score(y_test, y_prediction_candidate, zero_division=0),
        f1_score(y_test, y_prediction_candidate, zero_division=0)
    ]
})

threshold_comparison


In [ ]:
# 15. Save the trained model and results
os.makedirs("results", exist_ok=True)
os.makedirs("models", exist_ok=True)

model.save("models/malaria_cnn.keras")
results.to_csv("results/model_metrics.csv", index=False)

print("Model saved to: models/malaria_cnn.keras")
print("Metrics saved to: results/model_metrics.csv")


# 16. Final Project Summary

After running the notebook, use the actual values from the `results` table in your Thiranex report.

### Key points to explain

- The model performs binary classification of microscopic cell images.
- Images are resized and normalized before training.
- A CNN learns visual patterns from the training images.
- The confusion matrix shows correct and incorrect predictions by class.
- Precision, recall and F1-score provide more detail than accuracy alone.
- ROC-AUC evaluates the classifier across different probability thresholds.
- Threshold changes can affect false positives and false negatives.

### Limitation

This is an educational machine-learning project. It must not be presented as a clinical diagnostic system.
